# Exercise 2 — Inspect InstaNovo predictions & assemble nanobody nb6

**27666 · June 2026 · Module 2**

**Goal**: take real mass-spectrometry data from a nanobody experiment, look at what InstaNovo predicted at the peptide level, then reconstruct the full nanobody sequence using a **greedy assembly algorithm** we'll implement inline.

**Data**: `NB6.csv` — InstaNovo predictions on a nanobody (**nb6**) digested by 9 different proteases (Trypsin, Chymotrypsin, GluC, Elastase, ProteinaseK, Thermolysin, Papain, Krakatoa, Vesuvius) and measured by LC-MS/MS. Fetched via `wget` from the course repo — no manual upload required.

**Algorithm**: port of the greedy mode of [InstaNexus](https://github.com/Multiomics-Analytics-Group/InstaNexus). The full InstaNexus pipeline also runs MMseqs2 clustering + Clustal Omega alignment + logomaker consensus. We keep just the greedy scaffolding core and discuss the trade-offs at the end.

**Runtime**: ~5 min on free Colab. No native binaries, no heavy installs.


## Attribution

- **InstaNovo** (de novo peptide sequencing): Eloff K.\*, Kalogeropoulos K.\* et al. *Nat Mach Intell* 7, 565–579 (2025).
- **InstaNexus** (assembly pipeline whose greedy mode we re-implement): Reverenna M. et al. *Mol Cell Proteomics* 25:4 (2026).
  Greedy algorithm adapted from `instanexus/assembly.py` (MIT licence).
- **Nanobody nb6 reference sequence**: from `InstaNexus-main/json/sample_metadata.json` (DTU Biosustain / Bioengineering).
- **Raw MS data**: DTU Bioengineering Proteomics Core Facility, 2025.


## 1. Setup

Just plain Python this time. No native binaries, no large packages.


In [ ]:
# Pin pandas to Colab's bundled version to avoid resolver complaints.
!pip install -q 'pandas==2.2.2' biopython matplotlib seaborn tqdm

# NOTE: if pandas was already imported before this cell, restart the runtime
# (Runtime → Restart session) and re-run from the top.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_context('notebook')
WORK = Path('/content/27666_exercise2') if Path('/content').exists() else Path('exercise2_work')
WORK.mkdir(exist_ok=True)
print(f'working dir: {WORK}')

## 2. Get the data

`NB6.csv` lives in `Day_5/morning/data/` on the course repo. We download it via `wget` so students don't need to upload anything manually.


In [ ]:
SAMPLE_NAME = 'nb6'
NB_URL = 'https://raw.githubusercontent.com/DigBioLab/27666_Protein_Design/main/Day_5/morning/data/NB6.csv'
NB_PATH = WORK / f'{SAMPLE_NAME}.csv'

if not NB_PATH.exists():
    !wget -q $NB_URL -O $NB_PATH

assert NB_PATH.exists() and NB_PATH.stat().st_size > 1_000_000, (
    f'{NB_PATH} missing or suspiciously small — check the URL.'
)

df = pd.read_csv(NB_PATH)
print(f'sample: {SAMPLE_NAME}   file: {NB_PATH}   rows: {len(df):,}')
df.head()

## 3. Inspect the predictions

Columns:
- `experiment_name` — encodes the protease used in that run
- `scan_number` — index of the MS/MS scan
- `preds` — predicted peptide sequence (empty if InstaNovo had no confident prediction)
- `log_probs` — InstaNovo's log-probability of the prediction (sentinel `-1.0` means "no prediction")


In [ ]:
# Reference sequence of nb6 (from sample_metadata.json). Defined here so the
# pre-assembly coverage cell below can use it.
NB_REF = ('QVQLQESGGGLVQPGGSLRLSCTASLNIFSINAMGWYRQAPGKQRELVAAITSGGSTNYADSVKGRFTISRDNAKSTVY'
          'LQMNSLKPEDTAVYYCHAEGPFNIATKEQYDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH')
print(f'reference {SAMPLE_NAME}: {len(NB_REF)} aa')

In [ ]:
# Parse the protease out of the experiment name
def protease_of(name: str) -> str:
    m = re.search(r'Nanobodies_([A-Za-z]+)_\d+ng', name)
    return m.group(1) if m else 'unknown'

df['protease'] = df['experiment_name'].apply(protease_of)
df['has_pred'] = df['preds'].notna() & (df['preds'] != '')

summary = (df.groupby('protease')
             .agg(scans=('scan_number', 'count'),
                  with_pred=('has_pred', 'sum'),
                  hit_rate=('has_pred', 'mean'))
             .sort_values('scans', ascending=False))
print(summary)

In [ ]:
# log_probs distribution for real predictions only
real = df[df['has_pred'] & (df['log_probs'] > -1)].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(real['log_probs'], bins=60, ax=axes[0], color='steelblue')
axes[0].set(title='InstaNovo log-prob distribution (predicted peptides)',
             xlabel='log P', ylabel='# peptides')

sns.boxplot(data=real, x='protease', y='log_probs', ax=axes[1],
             order=summary.index.tolist(), color='lightsteelblue')
axes[1].set(title='Raw confidence by protease', xlabel='', ylabel='log P')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

**Observations to make**:
- A large fraction of scans have **no prediction** — that's normal: not every MS/MS scan is fragmenting a peptide cleanly.
- Confidence varies between proteases. Why? (Hint: tryptic peptides are over-represented in InstaNovo's training data.)
- The `log_probs` are *raw model log-likelihoods*, not calibrated probabilities. This is the FDR problem we mentioned on Module 2 slide 15.


## 4. Pre-assembly checks

Before running anything, get a feel for the data:
- **Per-residue confidence** on a 0–1 scale (instead of raw `log_probs`).
- **Peptide length** by protease.
- **Peptide overlap** between proteases (Jaccard).
- **Pre-assembly coverage** by exact substring match against the nb6 reference (the assembly-free baseline).


In [ ]:
# log_probs is the SUM of per-position log-probabilities. To compare peptides of
# different lengths, take the geometric mean per residue: exp(log_prob / L).
real['pep_len']     = real['preds'].str.len()
real['conf_per_aa'] = np.exp(real['log_probs'] / real['pep_len'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(real['conf_per_aa'], bins=60, ax=axes[0], color='steelblue')
axes[0].axvline(0.8, color='red', linestyle='--', label='cutoff 0.8')
axes[0].set(title='Per-residue confidence (0–1 scale)',
            xlabel='exp(log_p / L)', ylabel='# peptides')
axes[0].legend()

sns.boxplot(data=real, x='protease', y='conf_per_aa', ax=axes[1],
            order=summary.index.tolist(), color='lightsteelblue')
axes[1].axhline(0.8, color='red', linestyle='--')
axes[1].set(title='Per-residue confidence by protease', xlabel='', ylabel='exp(log_p / L)')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

n_hi = (real['conf_per_aa'] > 0.8).sum()
print(f'Peptides with per-residue conf > 0.8: {n_hi} ({n_hi/len(real):.1%})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=real, x='protease', y='pep_len', ax=ax,
            order=summary.index.tolist(), color='peachpuff')
ax.set(title='Peptide length distribution by protease', xlabel='', ylabel='peptide length (aa)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

print('Median peptide length per protease:')
print(real.groupby('protease')['pep_len'].median().sort_values(ascending=False))

In [ ]:
# Jaccard similarity of peptide sets between protease pairs.
# We WANT low off-diagonal Jaccard — different proteases should be complementary.
peps_by_protease = {p: set(real[real.protease == p].preds.unique())
                    for p in real.protease.unique()}
ps = sorted(peps_by_protease.keys())
n = len(ps)
J = np.zeros((n, n))
for i, p1 in enumerate(ps):
    for j, p2 in enumerate(ps):
        a, b = peps_by_protease[p1], peps_by_protease[p2]
        J[i, j] = len(a & b) / max(1, len(a | b))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(J, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=ps, yticklabels=ps, ax=ax,
            cbar_kws={'label': 'Jaccard similarity'})
ax.set_title('Peptide overlap between proteases')
plt.xticks(rotation=35); plt.tight_layout(); plt.show()

union = set().union(*peps_by_protease.values())
print(f'\nTotal unique peptides across all proteases: {len(union)}')
for p in ps:
    print(f'  {p:14s}: {len(peps_by_protease[p]):4d} unique peptides')

In [ ]:
# Quick coverage estimate: for each high-confidence peptide, find all exact substring
# matches against the nb6 reference. This is the assembly-free LOWER BOUND.
HIGH_CONF = real[real['conf_per_aa'] > 0.8]
covered_pre = np.zeros(len(NB_REF), dtype=int)
matched_peptides = 0
for pep in HIGH_CONF['preds'].unique():
    start, found = 0, False
    while True:
        idx = NB_REF.find(pep, start)
        if idx < 0: break
        covered_pre[idx:idx + len(pep)] += 1
        start = idx + 1
        found = True
    if found:
        matched_peptides += 1

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(NB_REF)), covered_pre, width=1.0, color='steelblue')
ax.set(xlabel=f'position in {SAMPLE_NAME} reference', ylabel='# peptide matches',
       title='Pre-assembly coverage by exact-match high-confidence peptides')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

frac = (covered_pre > 0).mean()
n_uniq_high = HIGH_CONF['preds'].nunique()
print(f'High-confidence unique peptides: {n_uniq_high}')
print(f'  ...matching the {SAMPLE_NAME} reference exactly: {matched_peptides}')
print(f'Reference positions covered by ≥1 high-confidence peptide: '
      f'{(covered_pre > 0).sum()} / {len(NB_REF)} ({frac:.1%})')
print('\nThis is a LOWER BOUND — assembly will chain peptides via overlapping ends.')

**What to look at**:
- **Per-residue confidence**: how many peptides survive a 0.8 cutoff? If very few, we should relax it.
- **Peptide length**: Trypsin / LysC give short, regular peptides; ProteinaseK / Elastase give a wider distribution. Mixing proteases is how we cover regions that one protease misses.
- **Overlap heatmap**: high off-diagonal Jaccard means two proteases are redundant. We *want* low off-diagonal values so the panel covers more of the protein.
- **Pre-assembly coverage**: rough preview of where assembly will succeed. If a region is dark here, the greedy chain has very little chance to recover it — and CDR3 is almost always the dark spot.


## 5. The greedy assembly algorithm

We re-implement the **greedy mode** of [InstaNexus](https://github.com/Multiomics-Analytics-Group/InstaNexus) inline so the notebook has no native-binary dependency. The full pipeline also runs MMseqs2 clustering + Clustal Omega alignment + a logomaker consensus step on top of greedy contigs — we skip those here to keep the moving parts small. Discussion at the end covers when those extra steps actually help.

**The core idea** (greedy assembly of a set of peptides):
1. For every pair of peptides (A, B), find the **largest suffix-of-A == prefix-of-B** of length ≥ `min_overlap`.
2. Sort all such overlaps by length, descending.
3. Greedily merge the highest-scoring pair first; each peptide can be used at most once per round.
4. Repeat until no more merges are possible (or until we hit `MAX_ITERATIONS`).

After per-protease assembly we get contigs. We then **scaffold** by pooling contigs across proteases and re-running the same greedy logic with substring deduplication between rounds.


In [ ]:
def find_sliding_overlaps(sequences, min_overlap):
    '''All pairs (i, j) with their largest suffix-of-i == prefix-of-j overlap (≥ min_overlap).'''
    overlaps = []
    for i, seq_a in enumerate(sequences):
        for j, seq_b in enumerate(sequences):
            if i == j:
                continue
            max_search = min(len(seq_a), len(seq_b))
            for length in range(max_search, min_overlap - 1, -1):
                if seq_a[-length:] == seq_b[:length]:
                    overlaps.append((i, j, length))
                    break   # only the LARGEST overlap per pair
    return overlaps


def merge_with_overhang(seq_a, seq_b, overlap_len):
    '''Merge a and b given suffix(a) == prefix(b). Handles substring containment.'''
    if seq_a in seq_b:
        return seq_b
    if seq_b in seq_a:
        return seq_a
    for i in range(len(seq_a) - overlap_len + 1):
        suffix = seq_a[i:]
        if seq_b.startswith(suffix):
            return seq_a[:i] + seq_b
    return seq_a + seq_b[overlap_len:]

In [ ]:
def assemble_contigs_greedy(peptides, min_overlap, max_iterations=50):
    '''Iterative greedy merging until convergence or max_iterations.'''
    contigs = list(peptides)
    for _ in range(max_iterations):
        overlaps = find_sliding_overlaps(contigs, min_overlap)
        if not overlaps:
            break
        overlaps.sort(key=lambda x: x[2], reverse=True)
        new_contigs = []
        used = set()
        for i, j, ovl in overlaps:
            if i in used or j in used:
                continue
            new_contigs.append(merge_with_overhang(contigs[i], contigs[j], ovl))
            used.update([i, j])
        if not new_contigs:
            break
        remaining = [c for k, c in enumerate(contigs) if k not in used]
        contigs = new_contigs + remaining
    return contigs


def remove_substring_contigs(contigs):
    '''Drop contigs that are substrings of longer contigs.'''
    contigs = sorted(set(contigs), key=len, reverse=True)
    keep = []
    for c in contigs:
        if not any(c != c2 and c in c2 for c2 in keep):
            keep.append(c)
    return keep

In [ ]:
def scaffold_iterative_greedy(contigs, min_overlap, size_threshold, max_rounds=2):
    '''Iteratively scaffold a pool of contigs into a smaller set of longer scaffolds.'''
    def clean(seqs):
        seqs = list(set(seqs))
        seqs = [s for s in seqs if len(s) > size_threshold]
        return sorted(seqs, key=len, reverse=True)

    current = clean(contigs)
    for round_idx in range(max_rounds):
        overlaps = find_sliding_overlaps(current, min_overlap)
        combined = [merge_with_overhang(current[i], current[j], ovl)
                    for i, j, ovl in overlaps]
        next_round = clean(remove_substring_contigs(combined + current))
        if len(next_round) == len(current):
            break
        current = next_round
    return current

## 6. Filter peptides and run the assembly on nb6

**Pipeline**: filter by confidence → drop contaminants → per-protease greedy assembly → cross-protease scaffolding.

We use a small **inline contaminants set** — trypsin self-digest, common keratin, BSA — typical lab MS contaminants. Any predicted peptide that is a substring of a contaminant sequence is discarded.


In [ ]:
# Hyperparameters — try changing these and re-running!
CONF_CUTOFF    = 0.8    # per-residue confidence cutoff for keeping a peptide
MIN_OVERLAP    = 3      # minimum AA overlap to chain two peptides
SIZE_THRESHOLD = 8      # discard contigs shorter than this in scaffolding

CONTAMINANTS = [
    # Trypsin self-digest fragment
    'VVGGYTCAANSIPYQVSLNSGSHFCGGSLINSQWVVSAAHCYKSRIQVRLGEHNIDVLEGNEQFINAAKIIKHPNFDR',
    # Human keratin K1 fragment
    'MSRQFSSRSGYRSGGGFSSGSAGIINYQRRTTSSSTRRSGGGGGRFSSCGGGGGSFGAGGGFGSRSLVNLGGSKSISIS',
    # BSA fragment
    'MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEEHFKGLVLIAFSQYLQQCPFDEHVKLVNELTEFAKTCVADESHAGCEK',
]

def is_contaminant(peptide):
    return any(peptide in c for c in CONTAMINANTS)

filtered = real[real['conf_per_aa'] > CONF_CUTOFF].copy()
n_before = len(filtered)
filtered = filtered[~filtered['preds'].apply(is_contaminant)]
n_after = len(filtered)
print(f'High-confidence peptides       : {n_before}')
print(f'... after contaminant removal  : {n_after}')
print(f'Discarded as contaminants      : {n_before - n_after}')

In [ ]:
# Per-protease greedy assembly
contigs_by_protease = {}
for protease, group in tqdm(filtered.groupby('protease'), desc='greedy per protease'):
    peps = group['preds'].unique().tolist()
    contigs = assemble_contigs_greedy(peps, min_overlap=MIN_OVERLAP)
    contigs = remove_substring_contigs(contigs)
    contigs_by_protease[protease] = contigs

print('\nContigs per protease (top 3 by length shown):')
for p, cs in contigs_by_protease.items():
    longest = sorted(cs, key=len, reverse=True)[:3]
    print(f'  {p:14s}: {len(cs):3d} contigs, longest={len(longest[0]) if longest else 0} aa')
    for c in longest:
        print(f'      ({len(c):3d}) {c[:80]}{"…" if len(c) > 80 else ""}')

In [ ]:
# Pool all contigs across proteases → scaffold iteratively
all_contigs = [c for cs in contigs_by_protease.values() for c in cs]
print(f'Pooled contigs (all proteases): {len(all_contigs)}')

scaffolds = scaffold_iterative_greedy(all_contigs,
                                       min_overlap=MIN_OVERLAP,
                                       size_threshold=SIZE_THRESHOLD)

print(f'\nFinal scaffolds: {len(scaffolds)}')
for i, s in enumerate(sorted(scaffolds, key=len, reverse=True)[:10]):
    print(f'  #{i+1}  ({len(s):3d} aa)  {s[:90]}{"…" if len(s) > 90 else ""}')

print(f'\n✓ Greedy assembly complete — {len(scaffolds)} scaffolds, longest {max(len(s) for s in scaffolds) if scaffolds else 0} aa.')

## 7. Map scaffolds onto the nb6 reference

Pairwise-align every scaffold against the known nb6 sequence, then plot a **coverage track** with the canonical VHH CDR1, CDR2 and CDR3 regions highlighted.


In [ ]:
from Bio import pairwise2

# Approximate CDR boundaries for VHH (Kabat-ish numbering applied to the framework).
# For production work use ANARCI or IMGT.
CDR1 = (25, 35)
CDR2 = (49, 58)
WGQ  = NB_REF.find('WGQGTQ')
CDR3 = (95, WGQ if WGQ > 0 else 115)

def coverage_mask(seqs):
    covered = np.zeros(len(NB_REF), dtype=int)
    for q in seqs:
        if not q: continue
        aln = pairwise2.align.localxs(NB_REF, q, -2, -1, one_alignment_only=True)
        if not aln: continue
        target_aln, query_aln, _, start, end = aln[0]
        ti = 0
        for tc, qc in zip(target_aln, query_aln):
            if tc != '-':
                if qc != '-' and ti >= start and ti < end:
                    covered[ti] += 1
                ti += 1
    return covered

scaffold_cov = coverage_mask(scaffolds)

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.bar(range(len(NB_REF)), scaffold_cov, width=1.0, color='#1f77b4')
ax.set_ylabel('# scaffolds covering')
ymax = max(1, ax.get_ylim()[1])
for cdr, name in zip([CDR1, CDR2, CDR3], ['CDR1', 'CDR2', 'CDR3']):
    ax.axvspan(cdr[0], cdr[1], color='red', alpha=0.12)
    ax.text((cdr[0]+cdr[1])/2, ymax*0.9, name, ha='center', fontsize=9, color='darkred')
ax.spines[['top','right']].set_visible(False)
ax.set_xlabel(f'position in {SAMPLE_NAME} reference')
ax.set_title(f'Greedy-assembly scaffold coverage of {SAMPLE_NAME}')
plt.tight_layout(); plt.savefig(WORK/f'{SAMPLE_NAME}_coverage.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'Reference length: {len(NB_REF)} aa')
print(f'Positions covered by ≥1 scaffold: {int((scaffold_cov > 0).sum())} ({(scaffold_cov > 0).mean():.1%})')

**Reading the track**:
- Tall bars = high redundancy at that position. Good.
- Dark gaps in CDR3 (the rightmost red band) are expected — that's the hypervariable loop.
- Dark gaps OUTSIDE the CDRs are bad: usually missed cleavage sites, low-signal protease, or contamination.


## 8. Best scaffold vs. reference — identity over the variable domain


In [ ]:
from Bio.pairwise2 import format_alignment

def best_pick(seqs):
    '''Return (seq, identity_over_VHH, alignment) of the scaffold with the highest
    identity to the nb6 variable domain (residues 0..WGQ).'''
    variable = NB_REF[:WGQ if WGQ > 0 else 120]
    best = ('', 0.0, None)
    for s in seqs:
        if not s: continue
        aln = pairwise2.align.globalxs(variable, s, -2, -1, one_alignment_only=True)
        if not aln: continue
        t, q, _, _, _ = aln[0]
        matches = sum(1 for a, b in zip(t, q) if a == b and a != '-')
        ident = matches / max(1, sum(1 for a in t if a != '-'))
        if ident > best[1]:
            best = (s, ident, aln[0])
    return best

best_seq, best_ident, best_aln = best_pick(scaffolds)
print(f'Best scaffold identity to {SAMPLE_NAME} variable domain: {best_ident:.1%}')
print(f'Scaffold length: {len(best_seq)} aa\n')
if best_aln:
    print(format_alignment(*best_aln, full_sequences=False)[:2400])

## 9. (Optional stretch) Close the loop with ESM-2

From Module 1: take the best scaffold and the true reference, embed both with ESM-2, compare. If the assembly is mostly right outside CDR3, the embeddings should be very close.


In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel

    MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).eval()

    @torch.no_grad()
    def embed(seq):
        x = tokenizer(seq, return_tensors='pt')
        h = model(**x).last_hidden_state[0, 1:-1].mean(0).numpy()
        return h / np.linalg.norm(h)

    ref_emb = embed(NB_REF)
    if best_seq:
        scaf_emb = embed(best_seq)
        cos = float(ref_emb @ scaf_emb)
        print(f'Cosine similarity of best scaffold to {SAMPLE_NAME} reference: {cos:.4f}')
        print('(1.0 = identical, 0.0 = orthogonal; expect 0.95+ on a successful assembly.)')
    else:
        print('No scaffold to compare.')
except Exception as e:
    print('Skipping ESM closure step:', e)

## 10. Discussion prompts

1. **Hyperparameter sweep**: try `CONF_CUTOFF ∈ {0.7, 0.8, 0.9}`, `MIN_OVERLAP ∈ {2, 3, 5}`, `SIZE_THRESHOLD ∈ {5, 8, 12}` and re-run from Section 6. Which combination maximises identity to the reference? Which produces the longest scaffold?
2. **Where did the assembly fail?** Look at the coverage track. Is CDR3 the worst region? Why is it so hard *both* for MS de novo *and* for MSA-based methods (Module 1 slide 10)?
3. **Which protease pulled its weight?** Look back at the per-protease contig list (Section 6). Which protease contributed the longest contigs? Which one underperformed? Could you have spent your beam time better?
4. **No reference?** In a real nanobody discovery campaign you don't *have* the reference. How would you judge assembly quality without one? (Hint: longest scaffold length, total covered residues, embedding plausibility from Module 1.)
5. **Pre-assembly checks (Section 4) vs. post-assembly result**: did the greedy chain close gaps that the exact-match preview missed? Where?
6. **What did we skip?** The full InstaNexus pipeline runs MMseqs2 clustering, Clustal Omega alignment and a logomaker consensus on top of greedy contigs. When would those extra steps help? Hint: multiple competing scaffolds covering the same region.

Bring one observation to the closing discussion.


## End of Exercise 2 — end of the 4-hour module

You have:
- Run a **foundation model** (ESM-2) on protein sequences and used the embeddings for visualisation, classification, zero-shot variant scoring, and a regression task (Exercise 1).
- **Inspected real InstaNovo predictions** on a nanobody MS dataset — log-prob distribution, per-residue confidence on a 0–1 scale, peptide length by protease, inter-protease Jaccard overlap, and a quick pre-assembly coverage check.
- **Implemented and ran greedy assembly** inline — per-protease contigs followed by cross-protease scaffolding — and seen how a ~100-line algorithm reconstructs much of a 153-aa nanobody from messy de novo peptide predictions.
- Compared the assembled scaffolds against the **known reference** and visualised coverage with CDR1/2/3 highlighted.
- **Closed the loop** by re-embedding the best scaffold with ESM-2 and measuring cosine similarity to the reference.

Same paradigm — foundation model pretraining → embeddings or scoring → downstream task — across three data modalities (protein sequences, mass spectra, and your own assembly output). That is the foundation-model paradigm in biology.
